# Stage 1–3 Evaluation Metrics (Updated)

This notebook explains the evaluation outputs you print in your training scripts:

- **Stage 1:** Text-only (TF‑IDF + Logistic Regression)  
- **Stage 2:** Metadata-only (DictVectorizer features + Logistic Regression)  
- **Stage 3:** Hybrid stacking (combines Stage 1 + Stage 2)

It focuses on what each metric means, how to interpret it, and what to mention in your report / supervisor meetings.


## 1) Quick recap: what the model outputs

### `preds = model.predict(X_test)`
- `predict(...)` returns **hard class labels** (0 or 1).
- In your project:
  - **0 = legit / safe**
  - **1 = phishing**

### `probs = model.predict_proba(X_test)[:, 1]` (if available)
- `predict_proba(...)` returns **probabilities** for each class.
- `[:, 1]` means: “probability the sample is class **1** (phishing)”.

You use probabilities for:
- **ROC‑AUC**
- **PR‑AUC**
- **best threshold search**
- confidence printing in `predict_*.py`


## 2) Core classification metrics

Assume:
- **TP (true positives):** phishing correctly predicted as phishing
- **TN (true negatives):** legit correctly predicted as legit
- **FP (false positives):** legit predicted as phishing
- **FN (false negatives):** phishing predicted as legit

### Accuracy
**Accuracy = (TP + TN) / (TP + TN + FP + FN)**  
How often the model is correct overall.

Good when classes are balanced; can be misleading if dataset is imbalanced.

### Precision (for phishing class)
**Precision = TP / (TP + FP)**  
When the model says “phishing”, how often it’s right.

High precision means **few false alarms** (few legit emails flagged).

### Recall (for phishing class)
**Recall = TP / (TP + FN)**  
Out of all real phishing emails, how many did we catch?

High recall means **few missed phishing emails**.

### F1 score
**F1 = 2 × (precision × recall) / (precision + recall)**  
Single score that balances precision and recall.

This is often used as a “best overall” metric for phishing detection because:
- false positives are annoying (precision)
- false negatives are dangerous (recall)


## 3) Confusion matrix and classification report

### Confusion matrix
`confusion_matrix(y_test, preds)` returns a 2×2 matrix (for binary classification):

```
[[TN, FP],
 [FN, TP]]
```

This lets you **see what type of mistakes** the model makes.

### Classification report
`classification_report(y_test, preds)` prints precision/recall/F1 for each class separately:
- class **0** (legit)
- class **1** (phishing)

In the report, it’s useful to quote:
- phishing class precision/recall/F1
- overall accuracy
- macro / weighted averages (optional)


## 4) ROC‑AUC and PR‑AUC (probability-based)

These metrics use **probabilities**, not hard 0/1 predictions.

### ROC‑AUC
- Measures how well the model ranks phishing above legit across thresholds.
- 0.5 = random, 1.0 = perfect.

Good general metric, but can look “too good” when data is easy or has leakage.

### PR‑AUC (Average Precision)
- Focuses on the positive class (phishing).
- More informative than ROC‑AUC when the dataset is imbalanced.

Rule of thumb:
- If phishing is rare in reality, PR‑AUC is often the more meaningful metric.


## 5) Threshold tuning (why “best threshold” exists)

By default, `predict(...)` uses a **0.5 threshold**:
- if phishing_prob ≥ 0.5 → predict phishing
- else → predict legit

But depending on your goal, you might choose a different threshold:
- higher threshold → fewer false positives (higher precision), but more missed phishing (lower recall)
- lower threshold → catch more phishing (higher recall), but more false alarms (lower precision)

Your code searches thresholds from 0→1 and picks the one that gives the **best F1** on the test split.
That’s what you print as:
- **Best threshold (by F1)**


## 6) Sanity check: label shuffle accuracy

You added a “label shuffle” check:

- Shuffle training labels (break the real relationship between X and y)
- Train again
- Accuracy should drop near chance (~0.5)

If shuffle accuracy stays high, that can indicate:
- leakage
- label encodes something trivial
- data generation artifacts

In your Stage 1 synthetic dataset, perfect scores were expected because the data was very “clean”.
On your real dataset, shuffle accuracy should be closer to 0.5–0.6, not 0.95+.


## 7) “Top features” output (interpretability)

For logistic regression with TF‑IDF:
- Each token/phrase gets a **coefficient**
- Positive coefficient → pushes prediction towards phishing (label=1)
- Negative coefficient → pushes prediction towards legit (label=0)

When you save `top_features.csv`, you’re exporting those strongest coefficients.

This is useful for the **implementation** section because you can show:
- “common phishing cues”: words like `click`, `verify`, `account`, etc.
- “common legit cues”: words like `thanks`, `meeting`, etc.

For Stage 2 (metadata), top features are things like:
- `has_ip_url`
- `upper_ratio`
- `exclam_count`
etc.


## 8) Why you can see “perfect” results (and why that can be suspicious)

Perfect 1.0 metrics can happen when:
- dataset is synthetic and too easy
- data has duplicates across train/test
- there is leakage (label correlates with a token that appears in both splits)
- the task is genuinely easy

What you did right:
- moved to a real dataset (`Phishing_Email.csv`)
- added label-shuffle check
- print confusion matrix + report
- added error analysis for Stage 2

In your report, you can explicitly say:
> “Synthetic baseline achieved near-perfect metrics due to clean separability; real dataset used for final comparison to avoid overestimating performance.”


## 9) Mini example (toy numbers)

Suppose on the test set:
- TP = 80
- TN = 900
- FP = 20
- FN = 40

Then:
- Accuracy = (80+900)/(80+900+20+40) = 980/1040 ≈ 0.942
- Precision = 80/(80+20) = 0.800
- Recall = 80/(80+40) = 0.667
- F1 = 2*(0.8*0.667)/(0.8+0.667) ≈ 0.727

Interpretation:
- good overall accuracy
- phishing precision ok (20 false alarms)
- recall is weaker (missed 40 phishing emails)
